# Bank Credit Risk Analysis
**Tools:** Python · Pandas · NumPy · Scikit-learn · XGBoost · Matplotlib · Seaborn · SciPy · Statsmodels  
**Goal:** End-to-end credit risk pipeline — EDA, statistical testing, customer segmentation, and default prediction model.

---

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import os

from scipy import stats
from scipy.stats import (ttest_ind, chi2_contingency, f_oneway, mannwhitneyu,
                          pearsonr, spearmanr, shapiro, levene, kruskal)
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                              RocCurveDisplay, ConfusionMatrixDisplay, roc_curve)
from sklearn.cluster import KMeans
from sklearn.inspection import permutation_importance
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

PLOTS_DIR = '../output/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

df = pd.read_csv('../data/credit_data_clean.csv')
print(f'Shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# 2.1 Descriptive Statistics
df.describe().T.style.background_gradient(cmap='Blues')

In [ ]:
# 2.2 Risk Label Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

risk_counts = df['risk_label'].value_counts()
colors = ['#2ecc71', '#f39c12', '#e74c3c']

axes[0].bar(risk_counts.index, risk_counts.values, color=colors, edgecolor='white', linewidth=1.2)
axes[0].set_title('Customer Risk Distribution', fontweight='bold')
axes[0].set_xlabel('Risk Label')
axes[0].set_ylabel('Count')
for i, (label, count) in enumerate(risk_counts.items()):
    axes[0].text(i, count + 20, str(count), ha='center', fontweight='bold')

axes[1].pie(risk_counts.values, labels=risk_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=140, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Risk Label Share', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/risk_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# 2.3 Feature Distributions
num_cols = ['age', 'income', 'loan_amount', 'credit_score',
            'credit_utilization', 'repayment_history', 'num_defaults', 'debt_to_income_ratio']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=30, color='#3498db', edgecolor='white', alpha=0.85)
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold')
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.5, label='Mean')
    axes[i].legend(fontsize=8)

plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/feature_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# 2.4 Box Plots by Risk Label
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
box_cols = ['credit_score', 'income', 'credit_utilization',
            'repayment_history', 'num_defaults', 'debt_to_income_ratio']

palette = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}
order = ['Low', 'Medium', 'High']

for i, col in enumerate(box_cols):
    sns.boxplot(data=df, x='risk_label', y=col, order=order,
                palette=palette, ax=axes[i], linewidth=1.2)
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold')
    axes[i].set_xlabel('')

plt.suptitle('Feature Distribution by Risk Label', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/boxplots_by_risk.png', bbox_inches='tight')
plt.show()

In [ ]:
# 2.5 Correlation Heatmap
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# 2.6 Credit Utilization vs Credit Score (Scatter)
fig, ax = plt.subplots(figsize=(9, 6))
palette = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}

for label in order:
    sub = df[df['risk_label'] == label]
    ax.scatter(sub['credit_utilization'], sub['credit_score'],
               label=label, alpha=0.4, s=15, color=palette[label])

ax.set_xlabel('Credit Utilization')
ax.set_ylabel('Credit Score')
ax.set_title('Credit Utilization vs Credit Score by Risk Label', fontweight='bold')
ax.legend(title='Risk Label')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/utilization_vs_score.png', bbox_inches='tight')
plt.show()

In [ ]:
# 2.7 Employment Status vs Risk
ct = pd.crosstab(df['employment_status'], df['risk_label'], normalize='index') * 100

ct[order].plot(kind='bar', stacked=True, figsize=(10, 5),
               color=['#2ecc71', '#f39c12', '#e74c3c'], edgecolor='white')
plt.title('Risk Distribution by Employment Status (%)', fontweight='bold')
plt.xlabel('Employment Status')
plt.ylabel('Percentage')
plt.xticks(rotation=15)
plt.legend(title='Risk Label')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/employment_vs_risk.png', bbox_inches='tight')
plt.show()

## 3. Statistical Testing (15+ Tests)

In [ ]:
low  = df[df['risk_label'] == 'Low']
mid  = df[df['risk_label'] == 'Medium']
high = df[df['risk_label'] == 'High']

results = []

def record(test, variable, stat, p, conclusion):
    results.append({'Test': test, 'Variable': variable,
                    'Statistic': round(stat, 4), 'p-value': round(p, 4),
                    'Significant (p<0.05)': 'Yes' if p < 0.05 else 'No',
                    'Conclusion': conclusion})

In [ ]:
# T1: Independent Samples t-test — Credit Score: Low vs High Risk
stat, p = ttest_ind(low['credit_score'], high['credit_score'])
record('Independent t-test', 'Credit Score (Low vs High)',
       stat, p, 'Significant difference in credit score between Low and High risk')

# T2: Independent Samples t-test — Income: Low vs High Risk
stat, p = ttest_ind(low['income'], high['income'])
record('Independent t-test', 'Income (Low vs High)',
       stat, p, 'Income differs significantly between Low and High risk groups')

# T3: Mann-Whitney U — Credit Utilization: Low vs High Risk (non-parametric)
stat, p = mannwhitneyu(low['credit_utilization'], high['credit_utilization'], alternative='two-sided')
record('Mann-Whitney U', 'Credit Utilization (Low vs High)',
       stat, p, 'Utilization distributions differ significantly')

# T4: One-Way ANOVA — Credit Score across all 3 risk groups
stat, p = f_oneway(low['credit_score'], mid['credit_score'], high['credit_score'])
record('One-Way ANOVA', 'Credit Score (all 3 groups)',
       stat, p, 'At least one group has significantly different mean credit score')

# T5: One-Way ANOVA — Income across all 3 risk groups
stat, p = f_oneway(low['income'], mid['income'], high['income'])
record('One-Way ANOVA', 'Income (all 3 groups)',
       stat, p, 'Income means differ significantly across risk groups')

# T6: Chi-Square — Employment Status vs Risk Label
ct_emp = pd.crosstab(df['employment_status'], df['risk_label'])
stat, p, dof, expected = chi2_contingency(ct_emp)
record('Chi-Square', 'Employment Status vs Risk Label',
       stat, p, 'Significant association between employment and risk')

# T7: Chi-Square — Num Defaults (0 vs 1+) vs Risk Label
df['has_default'] = (df['num_defaults'] > 0).astype(int)
ct_def = pd.crosstab(df['has_default'], df['risk_label'])
stat, p, dof, expected = chi2_contingency(ct_def)
record('Chi-Square', 'Has Default vs Risk Label',
       stat, p, 'Having a default is significantly associated with higher risk')

# T8: Pearson Correlation — Credit Score vs Credit Utilization
stat, p = pearsonr(df['credit_score'], df['credit_utilization'])
record('Pearson Correlation', 'Credit Score vs Utilization',
       stat, p, 'Negative correlation: higher score → lower utilization')

# T9: Spearman Correlation — DTI Ratio vs Risk Score Numeric
stat, p = spearmanr(df['debt_to_income_ratio'], df['risk_score_numeric'])
record('Spearman Correlation', 'DTI Ratio vs Risk Score',
       stat, p, 'Strong positive rank correlation between DTI and risk')

# T10: Shapiro-Wilk Normality — Credit Score (sample 200)
sample = df['credit_score'].sample(200, random_state=42)
stat, p = shapiro(sample)
record('Shapiro-Wilk', 'Credit Score (normality)',
       stat, p, 'Credit score distribution is approximately normal' if p > 0.05 else 'Not normally distributed')

# T11: Levene Test — Variance equality in credit score across risk groups
stat, p = levene(low['credit_score'], mid['credit_score'], high['credit_score'])
record('Levene Test', 'Credit Score variance across risk groups',
       stat, p, 'Variances are not equal across groups' if p < 0.05 else 'Variances are equal')

# T12: Kruskal-Wallis — Repayment History across risk groups (non-parametric ANOVA)
stat, p = kruskal(low['repayment_history'], mid['repayment_history'], high['repayment_history'])
record('Kruskal-Wallis', 'Repayment History across risk groups',
       stat, p, 'Repayment history differs significantly across risk groups')

# T13: Pearson Correlation — Income vs Loan Amount
stat, p = pearsonr(df['income'], df['loan_amount'])
record('Pearson Correlation', 'Income vs Loan Amount',
       stat, p, f'Correlation = {stat:.3f} between income and loan amount')

# T14: Spearman Correlation — Repayment History vs Num Defaults
stat, p = spearmanr(df['repayment_history'], df['num_defaults'])
record('Spearman Correlation', 'Repayment History vs Num Defaults',
       stat, p, 'Expected negative correlation between repayment and defaults')

# T15: Mann-Whitney U — DTI Ratio: High Risk vs Low Risk
stat, p = mannwhitneyu(high['debt_to_income_ratio'], low['debt_to_income_ratio'], alternative='greater')
record('Mann-Whitney U', 'DTI Ratio (High > Low Risk)',
       stat, p, 'High risk customers have significantly higher DTI ratio')

# T16: OLS Regression — Credit Score predicting Risk Score Numeric
X_ols = sm.add_constant(df['credit_score'])
model_ols = sm.OLS(df['risk_score_numeric'], X_ols).fit()
record('OLS Regression', 'Credit Score → Risk Score',
       model_ols.fvalue, model_ols.f_pvalue,
       f'Credit score is a significant predictor of risk (R²={model_ols.rsquared:.3f})')

results_df = pd.DataFrame(results)
results_df

In [ ]:
# OLS Summary
print(model_ols.summary())

## 4. Customer Segmentation — KMeans Clustering

In [ ]:
# 4.1 Elbow Method to find optimal K
cluster_features = ['credit_score', 'credit_utilization', 'repayment_history',
                     'num_defaults', 'debt_to_income_ratio', 'income']

scaler = StandardScaler()
X_cluster = scaler.fit_transform(df[cluster_features])

inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.title('Elbow Method — Optimal K', fontweight='bold')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.xticks(K_range)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/kmeans_elbow.png', bbox_inches='tight')
plt.show()

In [ ]:
# 4.2 Fit KMeans with K=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_cluster)

# Cluster profile
cluster_profile = df.groupby('cluster')[cluster_features + ['risk_score_numeric']].mean().round(2)
print('Cluster Profiles:')
cluster_profile

In [ ]:
# 4.3 Visualize clusters (Credit Score vs Credit Utilization)
fig, ax = plt.subplots(figsize=(9, 6))
cluster_colors = ['#3498db', '#e74c3c', '#2ecc71']

for c in sorted(df['cluster'].unique()):
    sub = df[df['cluster'] == c]
    ax.scatter(sub['credit_score'], sub['credit_utilization'],
               label=f'Cluster {c}', alpha=0.4, s=15, color=cluster_colors[c])

ax.set_xlabel('Credit Score')
ax.set_ylabel('Credit Utilization')
ax.set_title('KMeans Customer Segments', fontweight='bold')
ax.legend(title='Cluster')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/kmeans_clusters.png', bbox_inches='tight')
plt.show()

## 5. Credit Risk Prediction Model

In [ ]:
# 5.1 Feature Engineering for ML
le = LabelEncoder()
df['employment_encoded'] = le.fit_transform(df['employment_status'])

FEATURES = ['age', 'income', 'loan_amount', 'loan_tenure', 'credit_score',
            'credit_utilization', 'repayment_history', 'num_defaults',
            'debt_to_income_ratio', 'monthly_emi', 'emi_to_income_ratio', 'employment_encoded']

TARGET = 'risk_score_numeric'  # 0=Low, 1=Medium, 2=High

X = df[FEATURES]
y = df[TARGET]

print('Class distribution:')
print(pd.Series(y).value_counts())

In [ ]:
# 5.2 Handle Class Imbalance with SMOTE
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'Before SMOTE: {pd.Series(y_train).value_counts().to_dict()}')
print(f'After SMOTE:  {pd.Series(y_train_sm).value_counts().to_dict()}')

In [ ]:
# 5.3 Train Models

# Logistic Regression
scaler_ml = StandardScaler()
X_train_scaled = scaler_ml.fit_transform(X_train_sm)
X_test_scaled  = scaler_ml.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train_sm)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)

# XGBoost
xgb = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=5,
                     use_label_encoder=False, eval_metric='mlogloss',
                     random_state=42, n_jobs=-1)
xgb.fit(X_train_sm, y_train_sm)

print('Models trained: Logistic Regression, Random Forest, XGBoost')

In [ ]:
# 5.4 Model Evaluation
models = {'Logistic Regression': (lr, X_test_scaled), 
          'Random Forest': (rf, X_test), 
          'XGBoost': (xgb, X_test)}

for name, (model, X_eval) in models.items():
    y_pred = model.predict(X_eval)
    y_prob = model.predict_proba(X_eval)
    auc = roc_auc_score(y_test, y_prob, multi_class='ovr')
    print(f'\n--- {name} ---')
    print(f'ROC-AUC (OvR): {auc:.4f}')
    print(classification_report(y_test, y_pred, target_names=['Low', 'Medium', 'High']))

In [ ]:
# 5.5 Confusion Matrix — XGBoost
y_pred_xgb = xgb.predict(X_test)

fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_xgb,
    display_labels=['Low', 'Medium', 'High'],
    cmap='Blues', ax=ax
)
ax.set_title('XGBoost — Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/confusion_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
# 5.6 ROC-AUC Curves (One-vs-Rest for each class)
y_prob_xgb = xgb.predict_proba(X_test)
class_names = ['Low', 'Medium', 'High']
colors_roc = ['#2ecc71', '#f39c12', '#e74c3c']

fig, ax = plt.subplots(figsize=(8, 6))
for i, (cls, color) in enumerate(zip(class_names, colors_roc)):
    fpr, tpr, _ = roc_curve((y_test == i).astype(int), y_prob_xgb[:, i])
    auc_val = roc_auc_score((y_test == i).astype(int), y_prob_xgb[:, i])
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{cls} (AUC = {auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('XGBoost — ROC-AUC Curves (One-vs-Rest)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/roc_auc_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# 5.7 Feature Importance — XGBoost
importance = pd.Series(xgb.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
importance.plot(kind='barh', color='#3498db', edgecolor='white', ax=ax)
ax.set_title('XGBoost — Feature Importance', fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/feature_importance.png', bbox_inches='tight')
plt.show()

In [ ]:
# 5.8 Cross-Validation — XGBoost
cv_scores = cross_val_score(xgb, X, y, cv=StratifiedKFold(5), scoring='roc_auc_ovr', n_jobs=-1)
print(f'5-Fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# 5.9 VIF — Multicollinearity Check
X_vif = sm.add_constant(df[FEATURES])
vif_data = pd.DataFrame({
    'Feature': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
}).sort_values('VIF', ascending=False)

print('VIF > 10 indicates multicollinearity:')
vif_data[vif_data['Feature'] != 'const']

## 6. Business Insights Summary

In [ ]:
# Summary dashboard — key KPIs
print('=' * 55)
print('   BANK CREDIT RISK ANALYSIS — KEY FINDINGS')
print('=' * 55)
print(f"Total Customers Analyzed  : {len(df):,}")
print(f"Average Credit Score      : {df['credit_score'].mean():.1f}")
print(f"Average Income            : ${df['income'].mean():,.0f}")
print(f"Average Credit Utilization: {df['credit_utilization'].mean()*100:.1f}%")
print(f"High Risk Customers       : {(df['risk_label']=='High').sum():,} ({(df['risk_label']=='High').mean()*100:.1f}%)")
print(f"Total Loan Portfolio      : ${df['loan_amount'].sum():,.0f}")
print(f"High-Risk Loan Exposure   : ${df[df['risk_label']=='High']['loan_amount'].sum():,.0f}")
print()
print('Top predictors of default risk (XGBoost):')
for feat, imp in importance.tail(5)[::-1].items():
    print(f'  {feat:<30} {imp:.4f}')
print('=' * 55)